In [1]:
import sys

print("Python:", sys.version)
print("Ambiente:", sys.executable)

Python: 3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]
Ambiente: c:\Users\Dell\Documents\rocketseat\ai-customer-intelligence\.venv\Scripts\python.exe


In [4]:
import pandas as pd

print("Versão do Pandas:", pd.__version__)

Versão do Pandas: 3.1.1


In [3]:
%pip install pandas

  Using cached numpy-2.5.4-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
   ---------------------------------------- 0.0/10.9 MB ? eta -:--:--
   --------- ------------------------------ 2.6/10.9 MB 24.4 MB/s eta 0:00:01
   ---------------------------------------  10.7/10.9 MB 37.5 MB/s eta 0:00:01
   ---------------------------------------- 10.9/10.9 MB 32.9 MB/s  0:00:00
Using cached numpy-2.5.4-cp314-cp314-win_amd64.whl (12.7 MB)

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   -


[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from pathlib import Path

caminho = Path("../data/raw/orders.csv")

pedidos = pd.read_csv(caminho)

print("Linhas:", pedidos.shape[0])
print("Colunas:", pedidos.shape[1])

pedidos.head()

Linhas: 368999
Colunas: 29


,order_id,store_id,channel_id,payment_order_id,delivery_order_id,order_status,order_amount,order_delivery_fee,order_delivery_cost,order_created_hour,...,order_moment_delivering,order_moment_delivered,order_moment_finished,order_metric_collected_time,order_metric_paused_time,order_metric_production_time,order_metric_walking_time,order_metric_expediton_speed_time,order_metric_transit_time,order_metric_cycle_time
0,68405119,3512,5,68405119,68405119,CANCELED,62.7,0.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,68405123,3512,5,68405123,68405123,CANCELED,62.7,0.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,68405206,3512,5,68405206,68405206,CANCELED,115.5,0.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,68405465,3401,5,68405465,68405465,CANCELED,55.9,0.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,68406064,3401,5,68406064,68406064,CANCELED,37.9,0.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
pedidos.info()

<class 'pandas.DataFrame'>
RangeIndex: 368999 entries, 0 to 368998
Data columns (total 29 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   order_id                           368999 non-null  int64  
 1   store_id                           368999 non-null  int64  
 2   channel_id                         368999 non-null  int64  
 3   payment_order_id                   368999 non-null  int64  
 4   delivery_order_id                  368999 non-null  int64  
 5   order_status                       368999 non-null  str    
 6   order_amount                       368999 non-null  float64
 7   order_delivery_fee                 368999 non-null  float64
 8   order_delivery_cost                361794 non-null  float64
 9   order_created_hour                 368999 non-null  int64  
 10  order_created_minute               368999 non-null  int64  
 11  order_created_day                  368999 non-null

In [7]:
pedidos["order_status"].value_counts()

order_status
FINISHED    352020
CANCELED     16979
Name: count, dtype: int64

In [8]:
total_pedidos = len(pedidos)

pedidos_cancelados = (
    pedidos["order_status"] == "CANCELED"
).sum()

taxa_cancelamento = (
    pedidos_cancelados / total_pedidos
) * 100

print(f"Total de pedidos: {total_pedidos:,}")
print(f"Pedidos cancelados: {pedidos_cancelados:,}")
print(f"Taxa de cancelamento: {taxa_cancelamento:.2f}%")

Total de pedidos: 368,999
Pedidos cancelados: 16,979
Taxa de cancelamento: 4.60%


In [9]:
pedidos.groupby("order_status")[
    ["order_moment_delivered", "order_metric_cycle_time"]
].count()

,order_moment_delivered,order_metric_cycle_time
order_status,,
CANCELED,598,1723
FINISHED,19003,351657


In [10]:
colunas_tempo = [
    "order_moment_created",
    "order_moment_accepted",
    "order_moment_ready",
    "order_moment_collected",
    "order_moment_in_expedition",
    "order_moment_delivering",
    "order_moment_delivered",
    "order_moment_finished"
]

pedidos.groupby("order_status")[colunas_tempo].count().T

order_status,CANCELED,FINISHED
order_moment_created,16979,352020
order_moment_accepted,8886,350652
order_moment_ready,2916,340977
order_moment_collected,1849,324256
order_moment_in_expedition,1671,299899
order_moment_delivering,1317,342366
order_moment_delivered,598,19003
order_moment_finished,1383,352017


In [11]:
pedidos_finalizados = pedidos[
    pedidos["order_status"] == "FINISHED"
]

pedidos_finalizados["order_metric_cycle_time"].describe()

count    351657.000000
mean        155.843596
std        1225.229132
min           0.370000
25%          32.870000
50%          42.180000
75%          55.550000
max      154277.320000
Name: order_metric_cycle_time, dtype: float64

In [12]:
tempos = pedidos_finalizados["order_metric_cycle_time"]

q1 = tempos.quantile(0.25)
q3 = tempos.quantile(0.75)

iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

outliers = tempos[
    (tempos < limite_inferior) |
    (tempos > limite_superior)
]

print(f"Limite inferior: {limite_inferior:.2f}")
print(f"Limite superior: {limite_superior:.2f}")
print(f"Quantidade de outliers: {len(outliers):,}")
print(f"Percentual de outliers: {len(outliers) / tempos.count() * 100:.2f}%")

Limite inferior: -1.15
Limite superior: 89.57
Quantidade de outliers: 31,635
Percentual de outliers: 9.00%


In [13]:
tempos_sem_outliers = tempos[
    (tempos >= limite_inferior) &
    (tempos <= limite_superior)
]

print(f"Média original: {tempos.mean():.2f}")
print(f"Mediana original: {tempos.median():.2f}")

print(f"Média sem outliers: {tempos_sem_outliers.mean():.2f}")
print(f"Mediana sem outliers: {tempos_sem_outliers.median():.2f}")

Média original: 155.84
Mediana original: 42.18
Média sem outliers: 42.45
Mediana sem outliers: 40.47


### Análise dos valores atípicos no tempo de ciclo

Foram analisados 351.657 registros válidos de pedidos finalizados.

O método IQR identificou 31.635 valores atípicos (9,00%), com limite superior de 89,57.

A média original foi de 155,84, enquanto a mediana foi de 42,18. Após a exclusão temporária dos outliers, a média caiu para 42,45 e a mediana para 40,47.

Essa diferença indica forte influência dos valores extremos sobre a média.

**Decisão analítica:** preservar os registros originais, utilizar a mediana como referência do comportamento típico e investigar separadamente os valores extremos.

A unidade de medida e o significado operacional do tempo de ciclo ainda precisam ser confirmados.
